# gridkit helper v1
* gridkit installed in:
* /home/isatkaus/install/gridkit-manual/GridKit/

#### instructions
* /home/isatkaus/install/gridkit-manual/sh/kestrel_install.md

#### context
* /gridkit_docs 
* /home/isatkaus/.github/prompts/gridkit_docs.prompt.md
* has links to all gridkit doc .md files
* file is made every time we rebuild the gridkit mkdocs


In [4]:
lis = [
    "sdcs",
    "sdf",
    "sdf",
]

In [5]:
from IPython.display import Image

# import cufflinks as cl
import pandas as pd
import numpy as np
import os
import sys
import shutil
import re
import math
import json5
import datetime

import re
import textwrap

# import reV
# import PySAM


# import matplotlib.pyplot as plt
import glob
import os
import h5py
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp

# show multiple cell outputs
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

#### default values
n = pd.get_option("display.max_rows")
m = pd.get_option("display.max_columns")
k = pd.get_option("display.max_colwidth")

pd.set_option("display.max_rows", 10)  # 60 default
pd.set_option("display.max_columns", 100)  # 20 default
pd.set_option("display.max_colwidth", 60)  # 50 default


### use to use df.plot instead of df.iplot() - no need for cufflinks
pd.options.plotting.backend = "plotly"

import subprocess

In [3]:
# import geopandas as gpd
# import contextily as ctx
# import shapely

# # same as above but using shapely.Point
# from shapely.geometry import Point, LineString, Polygon

# # import numpy as np
# from powerscenarios.parser import Parser
# from powerscenarios.grid import Grid

# from ecmwfapi import ECMWFDataServer
# import netCDF4

# from rex import WindX
# from rex import Resource
# import rex
# import os

# # speeds up tab autocompletion significantly


# %config Completer.use_jedi = False

In [4]:
from IPython.display import Markdown, display


def printmd(string):
    display(Markdown(string))


printmd("**This text is bold and generated from a code cell.**")


def get_ipynb_name():
    try:
        ipynb_name = os.path.basename(globals()["__vsc_ipynb_file__"])
        # print(ipynb_name)
    except KeyError:
        ipynb_name = "unknown"
    return ipynb_name


ipynb_name = get_ipynb_name()
print(f"\nthis notebook name is: {ipynb_name}\n")


### helper to bold a row in a dataframe - useful for highlighting a specific row in a table
def bold_selected_row(df, row_idx):
    """Bold the row at the specified index in a DataFrame."""

    def highlight_row(row):
        return ["font-weight: bold" if row.name == row_idx else "" for _ in row]

    return df.style.apply(highlight_row, axis=1)


#### example usage:
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6], "C": [7, 8, 9]})
print("bold selected row")
bold_selected_row(df, 1)  # This will bold the second row (index 1)

**This text is bold and generated from a code cell.**


this notebook name is: gridkit_helper.ipynb

bold selected row


,A,B,C
0,1,4,7
1,2,5,8
2,3,6,9


# contents

[Jump to: basic plots](#basic-plots)

[Jump to: end](#end)

## gridkit shell setup
* build dir
* PATH
* import module

In [5]:
install_root_dir = "/home/isatkaus/install/gridkit-manual/GridKit/"
source_dir = os.path.join(install_root_dir, "GridKit/")

build_dir = os.path.join(install_root_dir, "build/")
# print(f"gridkit install_dir: {install_dir}")
print(f"gridkit build_dir: {build_dir}")

gridkit build_dir: /home/isatkaus/install/gridkit-manual/GridKit/build/


In [6]:
# runner = os.path.join(build_dir, "application/PhasorDynamics/PDSim")
runner = os.path.join(build_dir, "application/PhasorDynamics/DynamicSimulation")
print(
    f"Phasor Dynamics runner, DynamicSimulation, that takes in .solver.json as input:\n {runner}"
)

# Add DynamicSimulation to PATH so it can be called without the full path
os.environ["PATH"] = os.path.dirname(runner) + os.pathsep + os.environ["PATH"]
print(
    f"\nDynamicSimulation added to PATH. You can now run: DynamicSimulation <solver.json>"
)

Phasor Dynamics runner, DynamicSimulation, that takes in .solver.json as input:
 /home/isatkaus/install/gridkit-manual/GridKit/build/application/PhasorDynamics/DynamicSimulation

DynamicSimulation added to PATH. You can now run: DynamicSimulation <solver.json>


In [ ]:
!echo $PATH

In [ ]:
!module list

In [ ]:
# ####### load module to ! commands and subprocesses in this notebook
# #### not need to since we added -Wl,-rpath,${GCC13_LIBDIR} to the cmake flags in 6_build_gridkit.sh. This bakes the absolute path to the GCC 13 libstdc++.so directly into the ELF binary header of PDSim and all GridKit shared libs.

# import subprocess

# # Module to load — change this to load a different module
# module_name = "gcc-stdalone/13.1.0"

# # Problem: each ! shell command runs in its own subshell, so
# #   !module load gcc-stdalone/13.1.0
# # in one cell has no effect on the next cell — the module is gone.
# #
# # Solution: run "module load ... && env -0" in a subprocess to capture
# # the resulting environment, diff it against the baseline, and inject
# # only the *changed* variables into os.environ. All subsequent ! commands
# # and subprocess.run() calls inherit os.environ, so the GCC 13 libs
# # (mainly LD_LIBRARY_PATH) will be visible when PDSim is executed.
# #
# # env -0 uses null-delimited output so multi-line variable values parse safely.
# #
# # Idempotent: LOADEDMODULES is set by the module system and injected into
# # os.environ on first run, so re-running this cell is a no-op.

# if module_name in os.environ.get("LOADEDMODULES", ""):
#     print(f"{module_name} already loaded, skipping.")
# else:

#     def capture_env(extra_cmd=""):
#         """Run bash -c '<extra_cmd> && env -0' and return env as a dict."""
#         cmd = f"{extra_cmd + ' && ' if extra_cmd else ''}env -0"
#         result = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
#         env = {}
#         for entry in result.stdout.split("\0"):
#             if "=" in entry and not entry.startswith("BASH_FUNC"):
#                 k, _, v = entry.partition("=")
#                 env[k] = v
#         return env

#     env_before = capture_env()
#     env_after = capture_env(f"module load {module_name}")

#     if not env_after:
#         print(
#             f"module load failed — check that 'module' is available in a plain bash -c subshell"
#         )
#     else:
#         # Only inject variables that the module load actually changed/added
#         changed = {k: v for k, v in env_after.items() if env_before.get(k) != v}
#         for k, v in changed.items():
#             os.environ[k] = v
#         print(f"Injected {len(changed)} changed variable(s) from {module_name}:")
#         for k, v in changed.items():
#             print(f"  {k}={v[:80]}{'...' if len(v) > 80 else ''}")

In [ ]:
!module list


In [ ]:
### but build is done so that we're using the GCC 13 libs, so we should be good to run DynamicSimulation without loading the gcc-stdalone/13.1.0 module in this notebook
!which gcc
!which DynamicSimulation

# simple run: select case
* ThreeBusBasic, Hawaii, Illinois 


* <build_dir>/examples/PhasorDynamics/Tiny/ThreeBus/Basic/
* <build_dir>examples/PhasorDynamics/Medium/Hawaii/
* <build_dir>examples/PhasorDynamics/Large/Illinois/ 

In [51]:
#rel_dir= "examples/PhasorDynamics/Tiny/ThreeBus/Basic/"
rel_dir = "examples/PhasorDynamics/Medium/Hawaii/"
#rel_dir ="examples/PhasorDynamics/Large/Illinois/"
#rel_dir = "examples/PhasorDynamics/Tiny/ThreeBus/ZipLoad/"

#/home/isatkaus/install/gridkit-manual/GridKit/build/examples/PhasorDynamics/Tiny/ThreeBus/ZipLoad/ThreeBusZipLoad.json
example_source_dir = os.path.join(source_dir, rel_dir)
example_dir = os.path.join(build_dir, rel_dir)
print(f"example_source_dir: {example_source_dir}")
print(f"example_dir: {example_dir}")
!ls -l {example_dir}

example_source_dir: /home/isatkaus/install/gridkit-manual/GridKit/GridKit/examples/PhasorDynamics/Medium/Hawaii/
example_dir: /home/isatkaus/install/gridkit-manual/GridKit/build/examples/PhasorDynamics/Medium/Hawaii/
total 4833
drwxrwxr-x 2 isatkaus isatkaus   13312 Jun 10 08:07 CMakeFiles
-rw-rw-r-- 1 isatkaus isatkaus    1992 Jun 10 08:07 cmake_install.cmake
-rw-rw-r-- 1 isatkaus isatkaus     918 Jun 10 08:07 CTestTestfile.cmake
-rw-rw-r-- 1 isatkaus isatkaus  138773 Jun 10 08:07 hawaii.json
-rw-rw-r-- 1 isatkaus isatkaus     236 Jun 10 08:07 hawaii.solver.json
-rw-rw-r-- 1 isatkaus isatkaus    8388 Jun 10 08:07 Makefile
-rw-rw-r-- 1 isatkaus isatkaus 8564587 Jun 10 10:16 mon.csv


## to run in regular terminal shell:
`module load gcc-stdalone/13.1.0`

`export PATH="/home/isatkaus/install/gridkit-manual/GridKit/build/application/PhasorDynamics:$PATH"`

`cd /home/isatkaus/install/gridkit-manual/GridKit/build/examples/PhasorDynamics/Tiny/ThreeBus/Basic/`

`PDSim ThreeBusBasic.solver.json`



## to run in notebook subshell
* ! command
* subprocess

In [52]:
if rel_dir == "examples/PhasorDynamics/Tiny/ThreeBus/Basic/":
    solver_fn = "ThreeBusBasic.solver.json"
    case_fn = "ThreeBusBasic.case.json"
elif rel_dir == "examples/PhasorDynamics/Medium/Hawaii/":
    solver_fn = "hawaii.solver.json"
    case_fn = "hawaii.json"
elif rel_dir == "examples/PhasorDynamics/Large/Illinois/":
    solver_fn = "illinois.solver.json"
    case_fn = "illinois.json"

print(f"solver_fn: {solver_fn}")
print(f"case_fn:   {case_fn}")

test_run_dir = f"./test-runs/{case_fn.split('.')[0]}-v1/"
# test_run_dir = "/kfs2/projects/scidac/scidac-data/gridkit-runs/test-runs/illinois-v1/"

os.makedirs(test_run_dir, exist_ok=True)

# Copy ALL files from example_dir (solver, case, ref.csv, etc.)
copied = []
for fn in os.listdir(example_dir):
    src = os.path.join(example_dir, fn)
    if os.path.isfile(src):
        dst = os.path.join(test_run_dir, fn)
        shutil.copy(src, dst)
        copied.append(fn)

print(f"Copied {len(copied)} files to {os.path.abspath(test_run_dir)}:")
for fn in sorted(copied):
    print(f"  {fn}")

solver_fn: hawaii.solver.json
case_fn:   hawaii.json


'./test-runs/hawaii-v1/hawaii.json'

'./test-runs/hawaii-v1/cmake_install.cmake'

'./test-runs/hawaii-v1/hawaii.solver.json'

'./test-runs/hawaii-v1/mon.csv'

'./test-runs/hawaii-v1/CTestTestfile.cmake'

'./test-runs/hawaii-v1/Makefile'

Copied 6 files to /kfs2/projects/scidac/isatkaus/scidac-notebooks/test-runs/hawaii-v1:
  CTestTestfile.cmake
  Makefile
  cmake_install.cmake
  hawaii.json
  hawaii.solver.json
  mon.csv


# edit case files (optional)
* hawaii generator dispatch

In [53]:
# Import dispatch helpers from gridkit_utils
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import (
    get_case_path_for_editing,
    read_genrou_dispatch,
    plot_genrou_dispatch,
    patch_genrou_dispatch,
)

<module 'gridkit_utils' from '/kfs2/projects/scidac/isatkaus/scidac-notebooks/gridkit_utils.py'>

In [54]:
# Read and plot current Genrou dispatch defaults (from copied run dir case file)
case_path_for_edit = get_case_path_for_editing(test_run_dir, example_dir, case_fn)
case_name = case_fn.split(".")[0]
print(f"Case path for dispatch edits: {case_path_for_edit}")

dispatch_df = read_genrou_dispatch(case_path_for_edit)
print(f"Genrou rows: {len(dispatch_df)}")
dispatch_df

plot_genrou_dispatch(dispatch_df, case_name=case_name)

Case path for dispatch edits: ./test-runs/hawaii-v1/hawaii.json
Genrou rows: 39


,gen_id,bus,p0,q0
0,2_1,2,0.025000,0.008000
1,2_2,2,0.025000,0.008000
2,2_3,2,0.025000,0.008000
3,2_4,2,0.025000,0.008000
4,23_1,23,0.692747,0.000441
...,...,...,...,...
34,36_2,36,0.021010,0.010000
35,36_3,36,0.020000,0.010000
36,36_4,36,0.024800,0.010000
37,37_3,37,0.583242,0.056258


In [ ]:
### example patch: overwrite copied case file in test_run_dir
### edit this list for your aleatoric dispatch experiment
dispatch_updates = [
    {"id": "2_1", "p0": 0.030, "q0": 0.010},
    {"id": "23_1", "p0": 0.220, "q0": 0.070},
    {"id": "35_1", "p0": 0.250, "q0": 0.080},
]

changes_df = patch_genrou_dispatch(
    case_json_path=case_path_for_edit,
    updates=dispatch_updates,
    output_case_path=None,  # overwrite copied case file in run dir
)
changes_df

# quick verify after patch
dispatch_df_after = read_genrou_dispatch(case_path_for_edit)
dispatch_df_after[dispatch_df_after["gen_id"].isin([u["id"] for u in dispatch_updates])]

In [27]:
### run from test_run_dir (files were copied there and can be modified before this cell)
result = subprocess.run(
    ["DynamicSimulation", solver_fn],
    cwd=test_run_dir,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    import glob

    outputs = glob.glob(os.path.join(test_run_dir, "*.csv"))
    print(f"Output files: {outputs}")

Error Set:
  Bus_1_Vm:
    max     : 5.700000e-08 (at time 4.167e-03)
    L2-norm : 2.792418e-06
  Bus_2_Vm:
    max     : 6.797048e-01 (at time 9.988e+00)
    L2-norm : 1.060103e+01
  Bus_3_Vm:
    max     : 8.045064e-01 (at time 9.988e+00)
    L2-norm : 1.836679e+01
  Genrou_genrou_2_1_speed:
    max     : 1.534752e-02 (at time 1.000e+01)
    L2-norm : 1.238389e-01
  Genrou_genrou_3_1_speed:
    max     : 2.856320e+00 (at time 1.000e+01)
    L2-norm : 8.570701e+01
  Total:
    max     : 2.856320e+00 (at time 1.000e+01)
    L2-norm : 8.583767e+01
--- FAIL: Test monitor file vs reference file


Complete in 0.150929 seconds

STDERR: 


In [28]:
# # ### run via ! (output file is in example_dir)
# #!cd {example_dir} && PDSim ThreeBusBasic.solver.json
# !cd {example_dir} && DynamicSimulation {solver_fn}
# ### this has no solver.json
# #!cd {example_dir} && ./ThreeBusZipLoadJson

In [29]:
# ################# run via subprocess
# result = subprocess.run(
#     # [runner, "ThreeBusBasic.solver.json"],
#     [runner, "ThreeBusBasic.solver.json"],
#     cwd=example_dir,
#     capture_output=True,
#     text=True,
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)

## output
* DynamicSimulation should output mon.csv
* defined by "monitors" in ThreeBusBasic.case.json

#### mon.csv column naming:
* `Bus_<bus-name>_<quantity>` - `Bus_ALOHA138_Vr`
* `<MachineModel>_<bus-number>_<gen-id>_<state-or-output>` - `Genrou_34_5_delta`

In [30]:
# Bus_<bus-name>_<quantity>  - Bus_ALOHA138_Vr
# <MachineModel>_<bus-number>_<gen-id>_<state-or-output>  - Genrou_34_5_delta

In [31]:
# !ls -l {example_dir}

In [32]:
results_fn = os.path.join(test_run_dir, "mon.csv")
# mon_df = pd.read_csv(os.path.join(example_dir, results_fn),index_col=0)
### cwd
mon_df = pd.read_csv(results_fn, index_col=0)
mon_df

,Bus_1_Vm,Bus_2_Vm,Bus_3_Vm,Genrou_genrou_2_1_speed,Genrou_genrou_3_1_speed
t,,,,,
0.004167,1.06,1.095109,1.056264,1.000687,1.001264
0.008333,1.06,1.094632,1.056369,1.001350,1.002489
0.012500,1.06,1.094304,1.056611,1.001988,1.003669
0.016667,1.06,1.094098,1.056955,1.002602,1.004800
0.020833,1.06,1.093988,1.057373,1.003192,1.005875
...,...,...,...,...,...
9.983333,1.06,0.482898,0.696500,1.013411,3.852956
9.987500,1.06,0.380166,0.165438,1.013824,3.853540
9.991667,1.06,0.492982,0.644804,1.014272,3.854569


In [33]:
## plot all df columns vs time
fig = px.line(mon_df, x=mon_df.index, y=mon_df.columns)
fig.update_layout(
    title=f"{results_fn} output",
    xaxis_title="Time (s)",
    yaxis_title="Value",
    legend_title="Variables",
)

In [11]:
# Group and plot mon.csv by (element, variable) using MONITORABLE_VARS_BY_ELEMENT
from gridkit_utils import MONITORABLE_VARS_BY_ELEMENT
from collections import defaultdict

grouped_cols = defaultdict(list)
unmatched = []

for col in mon_df.columns:
    parts = col.split("_")
    if len(parts) < 3:
        unmatched.append(col)
        continue

    if parts[0] == "Bus":
        # Bus_<bus-name>_<quantity>
        element = "Bus"
        var = parts[-1]
    else:
        # <MachineModel>_<bus-number>_<gen-id>_<state-or-output>
        element = parts[0]
        var = parts[-1]

    if (
        element in MONITORABLE_VARS_BY_ELEMENT
        and var in MONITORABLE_VARS_BY_ELEMENT[element]
    ):
        grouped_cols[(element, var)].append(col)
    else:
        unmatched.append(col)

# one plot per (element, var), e.g. Bus-Vi, Bus-Vr, Genrou-delta, Genrou-omega
for element, var_list in MONITORABLE_VARS_BY_ELEMENT.items():
    for var in var_list:
        cols = grouped_cols.get((element, var), [])
        if not cols:
            continue

        fig = px.line(
            mon_df,
            x=mon_df.index,
            y=cols,
            title=f"{element} - {var} ({len(cols)} signal(s))",
            labels={"x": "Time (s)", "value": "Value", "variable": "Signal"},
        )
        _ = fig.update_layout(legend_title="Columns")
        _ = fig.show()

if unmatched:
    print("Unmatched columns:")
    for c in unmatched:
        print("  ", c)

# multiple runs for UQ

* 3-bus and hawaii cases: perturb H on multiple gen's
* perturb gen dispatch



## threebusbasic setup
* sampling H

In [ ]:
# === UQ config (ThreeBusBasic) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Tiny/ThreeBus/Basic/")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/threebusbasic-v1"
SERIALIZE_MODE = "stacked"  # "stacked" (single results.parquet) or "per_run" (run_NNN.parquet per run)
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 100
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides (applied to every run's .solver.json) ---
# None = use base case values unchanged
# tmax: simulation end time (s)
# events: list of {index, ...fields} — patch events by 0-based index
SOLVER_OVERRIDES = None
# SOLVER_OVERRIDES = {
#     "tmax": 20.0,
#     "events": [
#         {"index": 0, "time": 2.0},   # fault_on
#         {"index": 1, "time": 2.1},   # fault_off
#     ],
# }

# dist="uniform" with pct: lo = nominal*(1-pct), hi = nominal*(1+pct)
# dist="uniform" with lo/hi: fixed range
# dist="normal" with mean/std: Gaussian via LHS+ppf
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "genrou_2_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 2.7,
        "pct": 0.10,
    },
    {
        "class": "Genrou",
        "id": "genrou_3_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 1.6,
        "pct": 0.10,
    },
]

MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "infinite_bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR:    {BASE_CASE_DIR}")
print(f"UQ_RUN_ROOT:      {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:   {SERIALIZE_MODE}")
print(f"UQ_OUT_PATH:      {UQ_OUT_PATH}")
print(f"SOLVER_OVERRIDES: {SOLVER_OVERRIDES}")

## hawaii setup

In [ ]:
# === UQ config (Hawaii) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files (hawaii.json, not hawaii.case.json) ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Medium/Hawaii/")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v2"
SERIALIZE_MODE = "per_run"  # "stacked" (single results.parquet) or "per_run" (run_NNN.parquet per run)
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 1000
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides (applied to every run's .solver.json) ---
# None = use base case values unchanged
# tmax: simulation end time (s)
# events: list of {index, ...fields} — patch events by 0-based index
SOLVER_OVERRIDES = None
# SOLVER_OVERRIDES = {
#     "tmax": 20.0,
#     "events": [
#         {"index": 0, "time": 2.0},   # fault_on
#         {"index": 1, "time": 2.1},   # fault_off
#     ],
# }

# First generator on each of 4 different buses — diverse H values
# bus 2: H=3.69, bus 23: H=6.15, bus 34: H=4.35, bus 35: H=5.22
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "2_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 3.69,
        "pct": 0.10,
    },
    {
        "class": "Genrou",
        "id": "23_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 6.15,
        "pct": 0.10,
    },
    {
        "class": "Genrou",
        "id": "34_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 4.35,
        "pct": 0.10,
    },
    {
        "class": "Genrou",
        "id": "35_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 5.22,
        "pct": 0.10,
    },
]

MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR:    {BASE_CASE_DIR}")
print(f"UQ_RUN_ROOT:      {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:   {SERIALIZE_MODE}")
print(f"UQ_OUT_PATH:      {UQ_OUT_PATH}")
print(f"SOLVER_OVERRIDES: {SOLVER_OVERRIDES}")

In [ ]:
# === Write experiment metadata to meta.yml ===
import yaml

os.makedirs(UQ_RUN_ROOT, exist_ok=True)
meta = {
    "case": os.path.basename(BASE_CASE_DIR.rstrip("/")),
    "base_case_dir": BASE_CASE_DIR,
    "run_root": UQ_RUN_ROOT,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "n_samples": N_SAMPLES,
    "seed": SEED,
    "sample_method": SAMPLE_METHOD,
    "serialize_mode": SERIALIZE_MODE,
    "solver_overrides": SOLVER_OVERRIDES,
    "param_specs": PARAM_SPECS,
    "monitors_by_class": MONITORS_BY_CLASS,
}
meta_path = os.path.join(UQ_RUN_ROOT, "meta.yml")
with open(meta_path, "w") as f:
    yaml.dump(meta, f, default_flow_style=False, sort_keys=False)
print(f"Wrote {meta_path}")
print(open(meta_path).read())

In [ ]:
# === Generate samples ===
os.makedirs(UQ_RUN_ROOT, exist_ok=True)
samples_df = generate_samples(PARAM_SPECS, N=N_SAMPLES, seed=SEED, method=SAMPLE_METHOD)
samples_df.to_csv(os.path.join(UQ_RUN_ROOT, "samples.csv"))
samples_df

In [ ]:
# # === ( 3-bus case only) Visualize LHS samples in H1–H2 parameter space ===
#######
# col_h1, col_h2 = samples_df.columns[0], samples_df.columns[1]
# s0, s1 = PARAM_SPECS[0], PARAM_SPECS[1]
# lo1, hi1 = s0["nominal"] * (1 - s0["pct"]), s0["nominal"] * (1 + s0["pct"])
# lo2, hi2 = s1["nominal"] * (1 - s1["pct"]), s1["nominal"] * (1 + s1["pct"])

# fig = px.scatter(
#     samples_df.reset_index(),
#     x=col_h1,
#     y=col_h2,
#     text="index",
#     title=f"LHS samples (N={N_SAMPLES}) in H parameter space",
#     labels={col_h1: f"H₁  ({s0['id']})", col_h2: f"H₂  ({s1['id']})"},
# )
# _ = fig.update_traces(textposition="top center", marker_size=8)
# _ = fig.add_shape(
#     type="rect",
#     x0=lo1,
#     x1=hi1,
#     y0=lo2,
#     y1=hi2,
#     line=dict(color="gray", dash="dash"),
#     fillcolor="rgba(0,0,0,0)",
# )
# _ = fig.update_layout(
#     xaxis=dict(range=[lo1 - 0.05 * (hi1 - lo1), hi1 + 0.05 * (hi1 - lo1)]),
#     yaxis=dict(range=[lo2 - 0.05 * (hi2 - lo2), hi2 + 0.05 * (hi2 - lo2)]),
# )
# _ = fig.show()

In [ ]:
# === Create run dirs + patch case.json ===
for i, row in samples_df.iterrows():
    d = make_run_dir(
        BASE_CASE_DIR,
        UQ_RUN_ROOT,
        i,
        row,
        PARAM_SPECS,
        MONITORS_BY_CLASS,
        solver_overrides=SOLVER_OVERRIDES,
    )
    print(f"  run_{i:03d}: {d}")

In [ ]:
# === Run all samples ===
failed = []
for i, row in samples_df.iterrows():
    run_dir = os.path.join(UQ_RUN_ROOT, f"run_{i:03d}")
    result = run_sample(run_dir, runner)
    status = "OK" if result.returncode == 0 else "FAILED"
    print(f"  run_{i:03d}: {status}")
    if result.returncode != 0:
        failed.append(i)
        print("    STDERR:", result.stderr[:300])

print(f"\nDone. {N_SAMPLES - len(failed)}/{N_SAMPLES} succeeded.")

In [51]:
# === Collect results → Parquet ===
# Override SERIALIZE_MODE here to produce a different format without changing the config cell
SERIALIZE_MODE = "stacked"  # "stacked" or "per_run"
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

result = collect_and_save(UQ_RUN_ROOT, samples_df, UQ_OUT_PATH, mode=SERIALIZE_MODE)
if SERIALIZE_MODE == "stacked":
    results_df = result
    results_df
else:
    print(f"Written {len(result)} per-run files")
    for p in result[:5]:
        print(f"  {p}")
    if len(result) > 5:
        print(f"  ... ({len(result) - 5} more)")

Saved 2400000 rows (1000 runs) -> /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v2/results.parquet


,run_id,time,Bus_ALOHA138_Vm,Bus_ALOHA138_Va,Bus_ALOHA69_Vm,Bus_ALOHA69_Va,Bus_FLOWER69_Vm,Bus_FLOWER69_Va,Bus_WAVE69_Vm,Bus_WAVE69_Va,Bus_HONOLULU138_Vm,Bus_HONOLULU138_Va,Bus_HONOLULU69_Vm,Bus_HONOLULU69_Va,Bus_SURF69_Vm,Bus_SURF69_Va,Bus_KANEOHE69_Vm,Bus_KANEOHE69_Va,Bus_TURTLE138_Vm,Bus_TURTLE138_Va,Bus_TURTLE69_Vm,Bus_TURTLE69_Va,Bus_MAHALO69_Vm,Bus_MAHALO69_Va,Bus_LYCHEE69_Vm,Bus_LYCHEE69_Va,Bus_COCONUT69_Vm,Bus_COCONUT69_Va,Bus_KAILUA138_Vm,Bus_KAILUA138_Va,Bus_KAILUA69_Vm,Bus_KAILUA69_Va,Bus_PALM69_Vm,Bus_PALM69_Va,Bus_WAIMANALO69_Vm,Bus_WAIMANALO69_Va,Bus_VOLCANO69_Vm,Bus_VOLCANO69_Va,Bus_PEARL CITY69_Vm,Bus_PEARL CITY69_Va,Bus_MILILANI69_Vm,Bus_MILILANI69_Va,Bus_AIEA69_Vm,Bus_AIEA69_Va,Bus_WAIPAHU138_Vm,Bus_WAIPAHU138_Va,Bus_WAIPAHU69_Vm,Bus_WAIPAHU69_Va,Bus_KAPOLEI69_Vm,Bus_KAPOLEI69_Va,...,Genrou_27_1_delta,Genrou_27_1_omega,Genrou_27_2_delta,Genrou_27_2_omega,Genrou_28_1_delta,Genrou_28_1_omega,Genrou_28_2_delta,Genrou_28_2_omega,Genrou_33_1_delta,Genrou_33_1_omega,Genrou_34_1_delta,Genrou_34_1_omega,Genrou_34_2_delta,Genrou_34_2_omega,Genrou_34_3_delta,Genrou_34_3_omega,Genrou_34_4_delta,Genrou_34_4_omega,Genrou_34_5_delta,Genrou_34_5_omega,Genrou_34_6_delta,Genrou_34_6_omega,Genrou_35_1_delta,Genrou_35_1_omega,Genrou_35_2_delta,Genrou_35_2_omega,Genrou_35_4_delta,Genrou_35_4_omega,Genrou_35_5_delta,Genrou_35_5_omega,Genrou_35_7_delta,Genrou_35_7_omega,Genrou_35_8_delta,Genrou_35_8_omega,Genrou_36_1_delta,Genrou_36_1_omega,Genrou_36_2_delta,Genrou_36_2_omega,Genrou_36_3_delta,Genrou_36_3_omega,Genrou_36_4_delta,Genrou_36_4_omega,Genrou_37_3_delta,Genrou_37_3_omega,Genrou_37_5_delta,Genrou_37_5_omega,Genrou_2_1_H,Genrou_23_1_H,Genrou_34_1_H,Genrou_35_1_H
0,0,0.004167,0.993545,-0.019546,0.991225,-0.068546,0.984548,-0.082574,0.978800,-0.100284,0.988985,-0.036125,0.981206,-0.096485,0.980583,-0.098984,0.978619,-0.099525,0.986768,-0.044477,0.980941,-0.098645,0.974911,-0.113847,0.977327,-0.108182,0.978953,-0.106375,0.983764,-0.054737,0.978631,-0.105239,0.982187,-0.112679,0.975496,-0.121857,0.975960,-0.124288,0.991304,-0.061335,0.989660,-0.034277,0.979942,-0.091947,0.996397,-0.014740,1.000000,-0.036757,0.993402,-0.064697,...,0.480517,4.477521e-13,0.480175,4.455888e-13,0.429042,-4.523901e-12,0.444748,-4.570943e-12,0.329416,-3.121471e-12,0.904358,-1.990060e-12,0.818916,-2.134176e-12,0.818916,-2.134176e-12,0.818916,-2.134176e-12,0.816401,-2.133997e-12,0.904358,-2.139999e-12,0.939655,-1.106566e-12,0.938910,-1.120423e-12,0.938335,-1.120653e-12,0.828826,-1.132593e-12,0.939544,-1.120170e-12,0.608558,-1.149133e-12,0.505320,-2.879002e-12,0.527743,-2.850711e-12,0.505320,-2.879002e-12,0.605968,-2.740940e-12,0.659913,-4.753022e-12,0.787961,-4.739534e-12,4.047359,6.560280,4.678113,5.28400
1,0,0.008333,0.993545,-0.019546,0.991225,-0.068546,0.984548,-0.082574,0.978800,-0.100284,0.988985,-0.036125,0.981206,-0.096485,0.980583,-0.098984,0.978619,-0.099525,0.986768,-0.044477,0.980941,-0.098645,0.974911,-0.113847,0.977327,-0.108182,0.978953,-0.106375,0.983764,-0.054737,0.978631,-0.105239,0.982187,-0.112679,0.975496,-0.121857,0.975960,-0.124288,0.991304,-0.061335,0.989660,-0.034277,0.979942,-0.091947,0.996397,-0.014740,1.000000,-0.036757,0.993402,-0.064697,...,0.480517,4.578732e-13,0.480175,4.538599e-13,0.429042,-8.747901e-12,0.444748,-8.846280e-12,0.329416,-6.262815e-12,0.904358,-4.047478e-12,0.818916,-4.327599e-12,0.818916,-4.327599e-12,0.818916,-4.327599e-12,0.816401,-4.326873e-12,0.904358,-4.351542e-12,0.939655,-2.217039e-12,0.938910,-2.244765e-12,0.938335,-2.245228e-12,0.828826,-2.267274e-12,0.939544,-2.244254e-12,0.608558,-2.296347e-12,0.505320,-5.844479e-12,0.527743,-5.798705e-12,0.505320,-5.844479e-12,0.605968,-5.613596e-12,0.659913,-9.536411e-12,0.787961,-9.524330e-12,4.047359,6.560280,4.678113,5.28400
2,0,0.012500,0.993545,-0.019546,0.991225,-0.068546,0.984548,-0.082574,0.978800,-0.100284,0.988985,-0.036125,0.981206,-0.096485,0.980583,-0.098984,0.978619,-0.099525,0.986768,-0.044477,0.980941,-0.098645,0.974911,-0.113847,0.977327,-0.

In [52]:
# === results size: in-memory and on-disk ===
if SERIALIZE_MODE == "stacked":
    mem_mb = results_df.memory_usage(deep=True).sum() / 1024**2
    disk_mb = os.path.getsize(UQ_OUT_PATH) / 1024**2
    print(f"results_df:  {results_df.shape[0]:,} rows × {results_df.shape[1]} cols")
    print(f"  in-memory: {mem_mb:.1f} MB")
    print(f"  parquet on disk: {disk_mb:.2f} MB")
else:
    import glob

    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    total_disk_mb = sum(os.path.getsize(f) for f in run_files) / 1024**2
    print(f"per_run: {len(run_files)} files in {UQ_OUT_PATH}")
    print(
        f"  total on disk: {total_disk_mb:.2f} MB  ({total_disk_mb/len(run_files):.2f} MB/run)"
    )

results_df:  2,400,000 rows × 158 cols
  in-memory: 2893.1 MB
  parquet on disk: 2884.06 MB


# plotting 
* for sanity check only
* more involved plotting is in gridkit_viv.ipynb

In [ ]:
# === Sanity check plot: 2 runs × 1 bus signal + 1 gen signal per var ===
PLOT_MAX_RUNS = 2
PLOT_MAX_BUS_N = 1
PLOT_MAX_GEN_N = 1

from collections import defaultdict

rng_plot = np.random.default_rng(SEED)

param_cols = list(samples_df.columns)
skip_cols = {"run_id", "time", "Solver Status"} | set(param_cols)
mon_cols = [c for c in results_df.columns if c not in skip_cols]

all_run_ids = sorted(results_df["run_id"].unique())
plot_run_ids = list(
    rng_plot.choice(
        all_run_ids, size=min(PLOT_MAX_RUNS, len(all_run_ids)), replace=False
    )
)
print(f"Runs: {plot_run_ids}")
plot_df = results_df[results_df["run_id"].isin(plot_run_ids)]

col_groups = defaultdict(list)
for col in mon_cols:
    parts = col.split("_")
    key = ("Bus", parts[-1]) if parts[0] == "Bus" else (parts[0], parts[-1])
    col_groups[key].append(col)

for (elem, var), cols in sorted(col_groups.items()):
    max_n = PLOT_MAX_BUS_N if elem == "Bus" else PLOT_MAX_GEN_N
    plot_cols = list(rng_plot.choice(cols, size=min(max_n, len(cols)), replace=False))

    melted = plot_df[["time", "run_id"] + plot_cols].melt(
        id_vars=["time", "run_id"], var_name="signal", value_name=var
    )
    fig = px.line(
        melted,
        x="time",
        y=var,
        color="signal",
        line_group="run_id",
        title=f"{elem} {var} — {plot_cols}, {len(plot_run_ids)} runs",
        labels={"time": "Time (s)", var: var},
    )
    _ = fig.update_traces(opacity=0.7)
    _ = fig.update_layout(legend_title="signal")
    _ = fig.show()

In [ ]:
# # === Query parquet directly — run 3, Bus 2 Va ===
# import pyarrow.parquet as pq
# import pyarrow.compute as pc

# tbl = pq.read_table(UQ_OUT_PARQUET, filters=[("run_id", "=", 3)])
# va_cols = [c for c in tbl.schema.names if c.startswith("Bus_") and c.endswith("_Va")]
# run3_bus2_va = tbl.select(["time"] + va_cols).to_pandas()
# run3_bus2_va

# # === Query parquet — all runs, all Genrou omega signals ===
# tbl = pq.read_table(UQ_OUT_PARQUET)
# omega_cols = [c for c in tbl.schema.names if c.endswith("_omega")]
# all_omega = tbl.select(["run_id", "time"] + omega_cols).to_pandas()
# all_omega

# entire pipeline
* copy example to scidac-data/gridkit-runs
* modify .solver.json and .case.json if needed
* run PDSim on witht them
* plot results

In [ ]:
# copy example (.solver.json and .case.json) to scidac-data/gridkit-runs/ThreeBusBasic/
# /home/isatkaus/install/gridkit-manual/GridKit/build/examples/PhasorDynamics/Tiny/ThreeBus/Basic/
build_example_dir = os.path.join(
    build_dir, "examples/PhasorDynamics/Tiny/ThreeBus/Basic/"
)
run_example_dir = (
    "/home/isatkaus/projects/scidac/scidac-data/gridkit-runs/ThreeBusBasic/"
)


if not os.path.exists(run_example_dir):
    os.makedirs(run_example_dir)
shutil.copy(
    os.path.join(build_example_dir, "ThreeBusBasic.solver.json"),
    os.path.join(run_example_dir, "ThreeBusBasic.solver.json"),
)
shutil.copy(
    os.path.join(build_example_dir, "ThreeBusBasic.case.json"),
    os.path.join(run_example_dir, "ThreeBusBasic.case.json"),
)
print(f"Copied example files to\n {run_example_dir} \n")

In [ ]:
### read .solver.json into a dict so you can modify it before running PDSim
import json

solver_json_path = os.path.join(run_example_dir, "ThreeBusBasic.solver.json")
with open(solver_json_path, "r") as f:
    solver_config = json5.load(f)
print("Original solver_config:")
print(json.dumps(solver_config, indent=2))

In [ ]:
### run via subprocess in the new location, with output files also in the new location
result = subprocess.run(
    [runner, "ThreeBusBasic.solver.json"],
    cwd=run_example_dir,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# end